# SVD and Eigenspaces

Throughout, $A$ is $n \times k$ ($n$ rows, $k$ columns) unless it has to be square.

- **Part A.** The singular value decomposition $A = U\Sigma V^T$, which works for every matrix.
- **Part B.** Eigenvalues and eigenspaces of a square matrix.

In [1]:
import sympy as sp
import numpy as np
import scipy.linalg as sla        # NumPy has no LU; SciPy does
from IPython.display import display

**Matrix product: use `@`.**

| | SymPy | NumPy |
|---|---|---|
| `A @ B` | matrix product | matrix product |
| `A * B` | matrix product | **entry-by-entry** product |

To avoid mistakes, this notebook uses `@` everywhere.

In [2]:
X = np.array([[1, 2], [3, 4]])
print(X @ X)     # matrix product
print(X * X)     # entry by entry: NOT the matrix product

[[ 7 10]
 [15 22]]
[[ 1  4]
 [ 9 16]]


## Part A. Singular value decomposition (SVD)

The SVD works for **every** $n \times k$ matrix:
$$ \underset{n\times k}{A} = \underset{n\times n}{U}\ \underset{n\times k}{\Sigma}\ \underset{k\times k}{V^T}, $$
- $U$ and $V$ are **orthogonal**: $U^TU = I_n$, $V^TV = I_k$,
- $\Sigma$ is "diagonal" (zero except $\sigma_{11}, \sigma_{22}, \dots$) with entries $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$, the **singular values**. There are $\min(n, k)$ of them.

**How to find the singular values:** $A^TA$ is a $k\times k$ symmetric matrix. Its **eigenvalues** are the numbers $\lambda$ with
$$ \det(A^TA - \lambda I) = 0, $$
and they are all $\ge 0$. Then
$$ \sigma_i = \sqrt{\lambda_i}. $$

**Example ($2\times 3$).**
$$ A = \begin{pmatrix} 3 & 2 & 2 \\ 2 & 3 & -2 \end{pmatrix}. $$

In [3]:
A = sp.Matrix([[3, 2,  2],
               [2, 3, -2]])
AtA = A.T @ A
display(AtA)

lam = sp.symbols("lambda")
p = (AtA - lam*sp.eye(3)).det()
print("det(A^T A - lambda I) =", sp.factor(p))
print("roots:", sp.solve(p, lam))
print("same with eigenvals():", AtA.eigenvals())     # {eigenvalue: multiplicity}
print("singular values:      ", A.singular_values())

Matrix([
[13, 12,  2],
[12, 13, -2],
[ 2, -2,  8]])

det(A^T A - lambda I) = -lambda*(lambda - 25)*(lambda - 9)
roots: [0, 9, 25]
same with eigenvals(): {25: 1, 9: 1, 0: 1}
singular values:       [5, 3, 0]


The roots $25, 9, 0$ give $\sigma_1 = 5$, $\sigma_2 = 3$. The third root $0$ comes from $A^TA$ being $3\times 3$ while $A$ has only $\min(2, 3) = 2$ singular values. SymPy still lists it as $0$; NumPy returns only $\min(n, k)$ values.

**NumPy is the usual tool for the SVD:** `U, s, Vt = np.linalg.svd(A)`. Note it returns:
- `s` as a **1-D array** of singular values (largest first), not the matrix $\Sigma$,
- `Vt` $= V^T$, **already transposed**.

In [4]:
An = np.array([[3, 2, 2], [2, 3, -2]], dtype=float)
U, s, Vt = np.linalg.svd(An)
print("shapes: U", U.shape, "  s", s.shape, "  Vt", Vt.shape)
print("s =", s)

Sigma = np.zeros(An.shape)               # rebuild the n x k Sigma
Sigma[:len(s), :len(s)] = np.diag(s)
print("Sigma =\n", Sigma)
print("U @ Sigma @ Vt == A ?", np.allclose(U @ Sigma @ Vt, An))
print("U orthogonal?", np.allclose(U.T @ U, np.eye(2)), "  V orthogonal?", np.allclose(Vt @ Vt.T, np.eye(3)))

shapes: U (2, 2)   s (2,)   Vt (3, 3)
s = [5. 3.]
Sigma =
 [[5. 0. 0.]
 [0. 3. 0.]]
U @ Sigma @ Vt == A ? True
U orthogonal? True   V orthogonal? True


**Reduced SVD.** Only the first $m = \min(n, k)$ columns of $U$ and $V$ ever meet a nonzero $\sigma$. Dropping the rest gives
$$ \underset{n\times k}{A} = \underset{n\times m}{U}\ \underset{m\times m}{\Sigma}\ \underset{m\times k}{V^T}, $$
with $\Sigma$ now a square diagonal matrix. NumPy: `np.linalg.svd(A, full_matrices=False)`.

In [5]:
U, s, Vt = np.linalg.svd(An, full_matrices=False)
print("shapes: U", U.shape, "  Sigma", np.diag(s).shape, "  Vt", Vt.shape)
print("U @ diag(s) @ Vt == A ?", np.allclose(U @ np.diag(s) @ Vt, An))

shapes: U (2, 2)   Sigma (2, 2)   Vt (2, 3)
U @ diag(s) @ Vt == A ? True


**SymPy** has `U, S, V = A.singular_value_decomposition()` with $A = USV^T$. It returns a reduced form keeping only the **nonzero** $\sigma$'s, listed **smallest first**. Exact SVDs get messy quickly, so use SymPy mainly for `singular_values()`.

In [6]:
Us, Ss, Vs = A.singular_value_decomposition()
display(Ss)
print("U @ S @ V^T == A ?", sp.simplify(Us @ Ss @ Vs.T) == A)

Matrix([
[3, 0],
[0, 5]])

U @ S @ V^T == A ? True


**What the SVD tells you.**

- **Rank** = number of nonzero singular values. (Use a tolerance in NumPy: `np.sum(s > 1e-10)`.)
- **Determinant** (square $A$): $\det U, \det V = \pm 1$, so $|\det A| = \sigma_1 \sigma_2 \cdots \sigma_n$.
- **Geometry:** $A$ maps the unit circle/sphere to an ellipse/ellipsoid with semi-axes $\sigma_1, \sigma_2, \dots$. So $|\det A|$, the area/volume scale factor, is the product of the semi-axes.

In [7]:
Bn = np.array([[2, 0, 1], [1, 3, 2], [1, 1, 2]], dtype=float)
s = np.linalg.svd(Bn, compute_uv=False)     # singular values only
print("singular values:", s)
print("product:", np.prod(s), "   det:", np.linalg.det(Bn))

Sn = np.array([[1, 2, 3], [4, 5, 6], [5, 7, 9]], dtype=float)   # row 3 = row 1 + row 2
s = np.linalg.svd(Sn, compute_uv=False)
print("\nsingular values:", s, "  rank:", np.sum(s > 1e-10))

singular values: [4.56783625 1.91415026 0.68622209]
product: 6.000000000000002    det: 6.0

singular values: [1.56633231e+01 8.12593979e-01 1.06425071e-15]   rank: 2


## Part B. Eigenvalues and eigenspaces

From now on $A$ is **square**, $n\times n$. A nonzero vector $\mathbf{v}$ is an **eigenvector** of $A$ with **eigenvalue** $\lambda$ if
$$ A\mathbf{v} = \lambda \mathbf{v}. $$
Rewrite this as $(A - \lambda I)\mathbf{v} = \mathbf{0}$. A nonzero solution exists exactly when $A - \lambda I$ is singular:
$$ \det(A - \lambda I) = 0 \qquad (\text{the } \textbf{characteristic polynomial}, \text{ degree } n). $$

For each eigenvalue, the **eigenspace** is the null space
$$ E_\lambda = N(A - \lambda I) = \{\text{all eigenvectors for } \lambda\} \cup \{\mathbf{0}\}. $$
It is a subspace, so it is described by a **basis**.

Each eigenvalue has two multiplicities:
- **algebraic**: how many times $\lambda$ is a root of $\det(A - \lambda I)$,
- **geometric**: $\dim E_\lambda = n - \operatorname{rank}(A - \lambda I)$, the number of basis vectors of the eigenspace.

Always $1 \le \text{geometric} \le \text{algebraic}$.

**Example.**
$$ A = \begin{pmatrix} 3 & 1 & 1 \\ 0 & 2 & 0 \\ 1 & 1 & 3 \end{pmatrix}. $$

In [8]:
A = sp.Matrix([[3, 1, 1],
               [0, 2, 0],
               [1, 1, 3]])
lam = sp.symbols("lambda")
p = (A - lam*sp.eye(3)).det()
print("det(A - lambda I) =", sp.factor(p))
print("eigenvals():", A.eigenvals())          # {eigenvalue: algebraic multiplicity}

for ev in A.eigenvals():
    basis = (A - ev*sp.eye(3)).nullspace()     # basis of the eigenspace E_ev
    print(f"\nlambda = {ev}: geometric multiplicity {len(basis)}, basis:")
    display(*basis)

det(A - lambda I) = -(lambda - 4)*(lambda - 2)**2
eigenvals(): {2: 2, 4: 1}

lambda = 2: geometric multiplicity 2, basis:


Matrix([
[-1],
[ 1],
[ 0]])

Matrix([
[-1],
[ 0],
[ 1]])


lambda = 4: geometric multiplicity 1, basis:


Matrix([
[1],
[0],
[1]])

So $\lambda = 2$ has algebraic **and** geometric multiplicity $2$: $E_2$ is a plane. $\lambda = 4$ has multiplicity $1$: $E_4$ is a line.

**SymPy `eigenvects()`** does everything at once. It returns a list of triples
$$ (\lambda,\ \text{algebraic multiplicity},\ [\text{basis of } E_\lambda]). $$

In [9]:
for ev, alg, basis in A.eigenvects():
    print(f"lambda = {ev}: algebraic {alg}, geometric {len(basis)}")
    for v in basis:
        print("   v =", list(v), "   A v == lambda v ?", A @ v == ev * v)

lambda = 2: algebraic 2, geometric 2
   v = [-1, 1, 0]    A v == lambda v ? True
   v = [-1, 0, 1]    A v == lambda v ? True
lambda = 4: algebraic 1, geometric 1
   v = [1, 0, 1]    A v == lambda v ? True


**Any** nonzero vector of $E_\lambda$ is an eigenvector, not only the basis vectors. For example $2\mathbf{v}_1 - 3\mathbf{v}_2 \in E_2$:

In [10]:
v1, v2 = A.eigenvects()[0][2]      # the two basis vectors of E_2
w = 2*v1 - 3*v2
print("w =", list(w), "   A w == 2 w ?", A @ w == 2*w)

w = [1, 2, -3]    A w == 2 w ? True


**NumPy:** `w, V = np.linalg.eig(A)`.
- `w` is a 1-D array of eigenvalues, **repeated** by algebraic multiplicity, in **no particular order**.
- The **columns** of `V` are eigenvectors, scaled to length $1$: column `V[:, i]` goes with `w[i]`.
- One check covers all of them: $AV = V\operatorname{diag}(w)$.

**Pitfalls.**
1. For a repeated eigenvalue, NumPy picks **its own basis** of $E_\lambda$, usually different from SymPy's. Compare the *spaces*, not the vectors: check $(A - \lambda I)\mathbf{v} \approx \mathbf{0}$.
2. The result may come back **complex** (with `+0.j` parts) even when everything is real. If `np.allclose(w.imag, 0)`, use `w.real`.
3. For a **symmetric** matrix, use `np.linalg.eigh` instead (real, sorted ascending, orthonormal eigenvectors).

In [11]:
An = np.array(A, dtype=float)
w, V = np.linalg.eig(An)
print("w =", w)
print("V =\n", V)
print("A V == V diag(w) ?", np.allclose(An @ V, V @ np.diag(w)))

w, V = w.real, V.real                 # imaginary parts are all 0 here
for i in range(3):
    print(f"column {i}: lambda = {w[i]:.0f},  (A - lambda I) v = 0 ?",
          np.allclose((An - w[i]*np.eye(3)) @ V[:, i], 0))

w = [4.+0.j 2.+0.j 2.+0.j]
V =
 [[ 0.70710678+0.j -0.70710678+0.j -0.40824829+0.j]
 [ 0.        +0.j  0.        +0.j  0.81649658+0.j]
 [ 0.70710678+0.j  0.70710678+0.j -0.40824829+0.j]]


A V == V diag(w) ? True
column 0: lambda = 4,  (A - lambda I) v = 0 ? True
column 1: lambda = 2,  (A - lambda I) v = 0 ? True
column 2: lambda = 2,  (A - lambda I) v = 0 ? True


**When geometric < algebraic.** Not every matrix has enough eigenvectors.
$$ J = \begin{pmatrix} 2 & 1 \\ 0 & 2 \end{pmatrix}: \qquad \det(J - \lambda I) = (\lambda - 2)^2, \quad \text{but } E_2 = \operatorname{span}\{(1, 0)\}. $$
$\lambda = 2$ has algebraic multiplicity $2$ but geometric multiplicity only $1$. Such a matrix is called **defective**. NumPy still returns two columns, but they are (almost) **parallel**.

In [12]:
J = sp.Matrix([[2, 1],
               [0, 2]])
print("eigenvects():", J.eigenvects())

w, V = np.linalg.eig(np.array(J, dtype=float))
print("\nNumPy V =\n", V.real)
print("rank of V:", np.linalg.matrix_rank(V))     # 1: only one independent eigenvector

eigenvects(): [(2, 2, [Matrix([
[1],
[0]])])]

NumPy V =
 [[ 1.0000000e+00 -1.0000000e+00]
 [ 0.0000000e+00  4.4408921e-16]]
rank of V: 1


### Summary

$A$ is $n\times k$, $m = \min(n, k)$.

| | form | sizes | needs |
|---|---|---|---|
| **SVD** (full) | $A = U\Sigma V^T$ | $U$: $n\times n$; $\Sigma$: $n\times k$; $V$: $k\times k$ | **any** $A$ |
| **SVD** (reduced) | $A = U\Sigma V^T$ | $U$: $n\times m$; $\Sigma$: $m\times m$; $V$: $k\times m$ | **any** $A$ |

**Eigenvalues** (square $A$): roots of $\det(A - \lambda I)$. **Eigenspace** $E_\lambda = N(A - \lambda I)$. $1 \le$ geometric multiplicity $= \dim E_\lambda \le$ algebraic multiplicity. Singular values: $\sigma_i = \sqrt{\lambda_i(A^TA)}$.


## Exercises

Solve with **SymPy** (exact); check with **NumPy** where asked. Use `@` for matrix products.

**Syntax you will need:**

| task | SymPy | NumPy |
|---|---|---|
| make a matrix | `sp.Matrix([[1, 2], [3, 4]])` | `np.array([[1, 2], [3, 4]], dtype=float)` |
| matrix from a rule $a_{ij} = f(i, j)$ | `sp.Matrix(n, k, lambda i, j: ...)` | `np.fromfunction(lambda i, j: ..., (n, k))` |
| convert SymPy → NumPy | | `np.array(A, dtype=float)` |
| matrix product, transpose | `A @ B`, `A.T` | `A @ B`, `A.T` |
| identity | `sp.eye(n)` | `np.eye(n)` |
| determinant | `A.det()` | `np.linalg.det(A)` |
| characteristic polynomial | `(A - lam*sp.eye(n)).det()` | |
| symbol | `lam = sp.symbols("lambda")` | |
| factor | `sp.factor(expr)` | |
| eigenvalues | `A.eigenvals()` (dict: value → alg. mult.) | `np.linalg.eigvals(A)` |
| eigenvectors | `A.eigenvects()` (list of $(\lambda, \text{mult}, \text{basis})$) | `w, V = np.linalg.eig(A)`, columns `V[:, i]` |
| null space (basis) | `A.nullspace()` | |
| singular values | `A.singular_values()` | `np.linalg.svd(A, compute_uv=False)` |
| full SVD | | `U, s, Vt = np.linalg.svd(A)` |
| reduced SVD | `U, S, V = A.singular_value_decomposition()` ($A = USV^T$; note $V$, not $V^T$) | `np.linalg.svd(A, full_matrices=False)` |
| simplify | `sp.simplify(expr)` | |
| equal up to rounding | | `np.allclose(X, Y)` |

**Exercise 1 (SVD).** Let
$$ A = \begin{pmatrix} 2 & 2 \\ 1 & -1 \\ 2 & 2 \end{pmatrix}. $$

(a) In **SymPy**, compute $A^TA$ and its eigenvalues with `eigenvals()`, and take their square roots. Then compute the singular values of $A$ with `A.singular_values()`, and observe that they are the same numbers.

(b) Compute the **reduced** SVD with **NumPy**. Give the shapes of $U$, $\Sigma$, $V^T$, and check $U\Sigma V^T = A$.

(c) Compute the SVD exactly with **SymPy**: `U, S, V = A.singular_value_decomposition()`. Give the shapes of $U$, $S$, $V$, and check $USV^T = A$ (use `sp.simplify`). In what order does SymPy list the singular values, compared with NumPy?

In [13]:
# Exercise 1


**Exercise 2 (Eigenspaces).** Consider the two matrices
$$ M = \begin{pmatrix} m_{ij} \end{pmatrix}_{4\times 4}, \quad m_{ij} = (i + 1)(j + 1), \ \ i, j = 0, 1, 2, 3, \qquad\qquad N = \begin{pmatrix} 2 & 1 & 1 \\ 1 & 2 & 1 \\ 0 & 0 & 3 \end{pmatrix}. $$

For **each** of $M$ and $N$, use **SymPy** to find:

(a) the eigenvalues, with their **algebraic** multiplicities;

(b) the eigenvectors: a basis of each eigenspace $E_\lambda$;

(c) the **geometric** multiplicity of each eigenvalue. For which eigenvalue(s) is it smaller than the algebraic multiplicity?

*Hint:* build $M$ from the rule with `sp.Matrix(4, 4, lambda i, j: (i + 1)*(j + 1))`.

In [14]:
# Exercise 2
